In [ ]:
import os
import pickle
from openai import OpenAI  # pip install openai
import nltk
from nltk.tokenize import sent_tokenize
import os
from pprint import pprint

from mt_reasoning.utils import prompts_util, clients_util 
from tqdm import tqdm
import importlib

importlib.reload(prompts_util)
importlib.reload(clients_util)

nltk.download('punkt')


input_dict = {
    "text": "Your input text here",
    "min_words_per_sentence": 20,
    "target_min_words": 22,
    "target_max_words": 28

}

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5-mini") 
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "1.0"))

openai_client = OpenAI(api_key=OPENAI_API_KEY)


with open("data/extraction_pdf/outputs/grammer_info.pkl", "rb") as f:
    grammers_list = pickle.load(f)

chapter_category_map = {
    "1. Introduction": None,
    "2. Sketch of the Sociohistorical and Sociolinguistic": None,
    "3. Phonetics and Phonology": None,
    "4. Morphosyntax": ["Morphology", "Syntax"],
    "5. Selected Syntactic Characteristics": ["Syntax"],
    "6. Lexical Structures": ["Morphology"],
    "7. Language Variation and Change": ["Irregular_form"],
    "8. Conclusion": None
}



[nltk_data] Downloading package punkt to /home/snt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [14]:
def segment_by_fixed_size_with_step(text, group_size=3, step=2, language='english', keep_tail=False):
    sents = sent_tokenize(text, language=language)
    segments, idx_spans = [], []
    i = 0
    n = len(sents)
    while i < n:
        j = i + group_size
        if j <= n:
            seg = " ".join(sents[i:j])
            segments.append(seg)
            idx_spans.append((i, j))
        elif keep_tail and i < n:
            seg = " ".join(sents[i:n])
            segments.append(seg)
            idx_spans.append((i, n))
            break
        else:
            break
        i += step
    return segments, idx_spans, sents

In [15]:


for grammers in grammers_list:
    chapter = grammers['section_title']
    print(chapter)
    if chapter in chapter_category_map and chapter_category_map[chapter] and chapter == "5. Selected Syntactic Characteristics":
        content_dict_list = grammers["content_dict_list"]
        for content_dict in content_dict_list:
            if content_dict["dtype"] == "text":
                test_input = content_dict["text"]
                print(test_input)
                break
        break



segments, spans, sents = segment_by_fixed_size_with_step(
    test_input, group_size=6, step=3, language='english', keep_tail=False
)


Luxembourgish
1. Introduction
2. Sketch of the Sociohistorical and Sociolinguistic
Evolution
3. Phonetics and Phonology
4. Morphosyntax
5. Selected Syntactic Characteristics
 
47 
wäerten 
wäert 
- 
- 
‘to become’ 
 
When used as infinite forms in verbal clusters, the modal verbs have recently developed 
an infixed -t- (combined sometimes with vowel alternation), probably originating from a 
subjunctive or preterit form (so-called supine; see Dammel, 2006, p. 154; Döhmer, 
2020). The occurrence of the supine forms like däerften (< däerfen), kéinten/kinnten (< 
kënnen), missten (< mussen) is variable, and its functions are not fully analyzed yet. 
According to the corpus analysis by Döhmer (2020, p. 213), they occur in between 10% 
and 50% of the verbal clusters. 
 
(41) Supine forms of the modal verbs  
Mir hunn däerften am Knascht spillen 
‘We were allowed to play in the dirt.’ 
Dat hätt missten intern beschwat ginn. 
‘This should have been discussed internally.’ 
Du häss kinnte(n) sc

In [ ]:

for m, (seg, (a, b)) in enumerate(zip(segments, spans), 1):
    # Update the input_dict with the current segment
    input_dict["text"] = seg
    print(f"{m:02d} [{a+1}-{b}]: {seg}")
    print("----------------------------------------------")
    obj = clients_util.generate_with_calling_openai_api(
                        client=openai_client,
                        system_prompt_template_path = "prompts/system/system_prompt_translation.jinja",         
                        input_prompt_template_path = "prompts/extraction/prompt_extraction_with_grammer_text.jinja",
                        input_text_dict = input_dict,
                        model = "gpt-5-mini",
    )
    print(obj)
    print("----------------------------------------------")
    pprint(obj, indent=2, width=150, sort_dicts=False)
    # break


01 [1-6]:  
47 
wäerten 
wäert 
- 
- 
‘to become’ 
 
When used as infinite forms in verbal clusters, the modal verbs have recently developed 
an infixed -t- (combined sometimes with vowel alternation), probably originating from a 
subjunctive or preterit form (so-called supine; see Dammel, 2006, p. 154; Döhmer, 
2020). The occurrence of the supine forms like däerften (< däerfen), kéinten/kinnten (< 
kënnen), missten (< mussen) is variable, and its functions are not fully analyzed yet. According to the corpus analysis by Döhmer (2020, p. 213), they occur in between 10% 
and 50% of the verbal clusters. (41) Supine forms of the modal verbs  
Mir hunn däerften am Knascht spillen 
‘We were allowed to play in the dirt.’ 
Dat hätt missten intern beschwat ginn. ‘This should have been discussed internally.’ 
Du häss kinnte(n) schwamme(n) goen. ‘You could have gone for a swim.’ 
 
 
5.
----------------------------------------------
{'grammars': [{'grammar_1': 'Supine (infixed -t-) forms of modal

KeyboardInterrupt: 